# Air Traffic Data Analysis - Student Exercise
## Inferential Statistics and Regression Analysis

**Student Template - Complete the TODO sections**

In this exercise, you will analyze air traffic data using inferential statistics and regression techniques. Follow the instructions and complete each section marked with `#TODO`.

### Dataset Description:
- **Dom_Pax**: Domestic Air Travel Passengers
- **Int_Pax**: International Air Travel Passengers  
- **Pax**: Total Air Travel Passengers
- **Dom_Flt**: Number of Flights (Domestic)
- **Int_Flt**: Number of Flights (International)
- **Flt**: Number of Flights (Total)
- **Dom_RPM**: Revenue Passenger-miles (Domestic)

## 1. Setup and Data Loading

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")


In [ ]:
# Load the dataset
try:
    df = pd.read_csv('air_traffic_data.csv')
    print("Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print("Creating sample air traffic data...")
    import numpy as np
    import pandas as pd
    
    # Create sample data
    np.random.seed(42)
    n_samples = 200
    
    # Generate correlated data
    dom_flights = np.random.normal(15000, 3000, n_samples)
    int_flights = np.random.normal(8000, 2000, n_samples)
    
    dom_pax = dom_flights * np.random.normal(12, 2, n_samples) + np.random.normal(0, 10000, n_samples)
    int_pax = int_flights * np.random.normal(15, 3, n_samples) + np.random.normal(0, 15000, n_samples)
    
    dom_rpm = dom_pax * np.random.normal(800, 100, n_samples)
    
    # Ensure positive values
    dom_flights = np.abs(dom_flights)
    int_flights = np.abs(int_flights)
    dom_pax = np.abs(dom_pax)
    int_pax = np.abs(int_pax)
    dom_rpm = np.abs(dom_rpm)
    
    df = pd.DataFrame({
        'Dom_Flt': dom_flights.astype(int),
        'Int_Flt': int_flights.astype(int),
        'Flt': (dom_flights + int_flights).astype(int),
        'Dom_Pax': dom_pax.astype(int),
        'Int_Pax': int_pax.astype(int),
        'Pax': (dom_pax + int_pax).astype(int),
        'Dom_RPM': dom_rpm.astype(int)
    })
    
    print("Sample data created successfully!")
    print(f"Shape: {df.shape}")


## 2. Exploratory Data Analysis

In [ ]:
# Display basic information about the dataset
print("Dataset Info:")
df.info()

print("\nFirst 5 rows:")
print(df.head())

print("\nBasic Statistics:")
print(df.describe())


In [ ]:
# Check for missing values and handle them if necessary
print("Missing values:")
print(df.isnull().sum())

if df.isnull().sum().sum() > 0:
    print("\nHandling missing values...")
    df = df.dropna()
    print(f"New shape after handling missing values: {df.shape}")


In [ ]:
# Create and analyze correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, square=True, linewidths=0.5)

plt.title('Correlation Matrix of Air Traffic Variables')
plt.tight_layout()
plt.show()

print("Strongest correlations:")
corr_pairs = correlation_matrix.unstack()
strong_pairs = corr_pairs[(abs(corr_pairs) > 0.7) & (corr_pairs < 1.0)].drop_duplicates()
print(strong_pairs.sort_values(ascending=False))


## 3. Hypothesis Testing

In [ ]:
# Hypothesis Test 1 - Compare domestic and international passengers
print("Hypothesis Test 1: Domestic vs International Passengers")
print("H0: Mean domestic passengers = Mean international passengers")
print("H1: Mean domestic passengers ≠ Mean international passengers")
print("Significance level: α = 0.05")

t_stat, p_value = stats.ttest_ind(df['Dom_Pax'], df['Int_Pax'])

print(f"\nResults:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")

print(f"Mean Domestic Passengers: {df['Dom_Pax'].mean():.0f}")
print(f"Mean International Passengers: {df['Int_Pax'].mean():.0f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nConclusion: Reject H0 (p < {alpha})")
    print("There is a statistically significant difference between the mean number of domestic and international passengers.")
else:
    print(f"\nConclusion: Fail to reject H0 (p >= {alpha})")
    print("There is no statistically significant difference between the mean number of domestic and international passengers.")


In [ ]:
# Hypothesis Test 2 - Test correlation between total passengers and total flights
print("\nHypothesis Test 2: Correlation between Total Passengers and Total Flights")
print("H0: There is no correlation between total passengers and total flights (ρ = 0)")
print("H1: There is a correlation between total passengers and total flights (ρ ≠ 0)")
print("Significance level: α = 0.05")

correlation_coef, p_value_corr = stats.pearsonr(df['Pax'], df['Flt'])

print(f"\nResults:")
print(f"Correlation coefficient: {correlation_coef:.4f}")
print(f"P-value: {p_value_corr:.6f}")

if p_value_corr < alpha:
    print(f"\nConclusion: Reject H0 (p < {alpha})")
    print(f"There is a significant correlation between total passengers and total flights.")
    if correlation_coef > 0:
        print("This is a positive correlation, meaning as the number of flights increases, the total number of passengers also tends to increase.")
    else:
        print("This is a negative correlation, meaning as the number of flights increases, the total number of passengers tends to decrease.")
else:
    print(f"\nConclusion: Fail to reject H0 (p >= {alpha})")
    print("We do not have enough evidence to claim a significant linear correlation between total passengers and total flights.")


## 4. Simple Linear Regression

In [ ]:
# Build a simple linear regression model
print("Simple Linear Regression: Predicting Total Passengers from Total Flights")

X_simple = df[['Flt']]
y_simple = df['Pax']

X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(X_simple, y_simple, test_size=0.2, random_state=42)

simple_model = LinearRegression()
simple_model.fit(X_train_simple, y_train_simple)

y_pred_simple = simple_model.predict(X_test_simple)

r2_simple = r2_score(y_test_simple, y_pred_simple)
mse_simple = mean_squared_error(y_test_simple, y_pred_simple)
mae_simple = mean_absolute_error(y_test_simple, y_pred_simple)
rmse_simple = np.sqrt(mse_simple)

print(f"\nModel Performance:")
print(f"R² Score: {r2_simple:.4f}")
print(f"Mean Squared Error: {mse_simple:.2f}")
print(f"Root Mean Squared Error: {rmse_simple:.2f}")
print(f"Mean Absolute Error: {mae_simple:.2f}")

print(f"\nModel Equation: Passengers = {simple_model.intercept_:.2f} + {simple_model.coef_[0]:.2f} × Flights")


In [ ]:
# Visualize the simple linear regression results
plt.figure(figsize=(12, 5))

# Plot 1: Scatter plot with regression line
plt.subplot(1, 2, 1)
plt.scatter(X_test_simple, y_test_simple, color='blue', alpha=0.5, label='Actual')
plt.plot(X_test_simple, y_pred_simple, color='red', linewidth=2, label='Predicted Line')
plt.xlabel('Total Flights')
plt.ylabel('Total Passengers')
plt.title('Simple Regression: Passengers vs Flights')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Residual plot
plt.subplot(1, 2, 2)
residuals = y_test_simple - y_pred_simple
plt.scatter(y_pred_simple, residuals, color='green', alpha=0.5)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot - Simple Linear Regression')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Multiple Linear Regression

In [ ]:
# Build a multiple linear regression model
print("Multiple Linear Regression: Predicting Total Passengers from Multiple Features")

feature_columns = ['Dom_Pax', 'Int_Pax', 'Dom_Flt', 'Int_Flt', 'Dom_RPM']
X_multiple = df[feature_columns]
y_multiple = df['Pax']

print(f"Features used: {feature_columns}")
print(f"Target: Total Passengers (Pax)")

X_train_mult, X_test_mult, y_train_mult, y_test_mult = train_test_split(X_multiple, y_multiple, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_mult_scaled = scaler.fit_transform(X_train_mult)
X_test_mult_scaled = scaler.transform(X_test_mult)

multiple_model = LinearRegression()
multiple_model.fit(X_train_mult_scaled, y_train_mult)

y_pred_mult = multiple_model.predict(X_test_mult_scaled)

r2_mult = r2_score(y_test_mult, y_pred_mult)
mse_mult = mean_squared_error(y_test_mult, y_pred_mult)
mae_mult = mean_absolute_error(y_test_mult, y_pred_mult)
rmse_mult = np.sqrt(mse_mult)

print(f"\nModel Performance:")
print(f"R² Score: {r2_mult:.4f}")
print(f"Mean Squared Error: {mse_mult:.2f}")
print(f"Root Mean Squared Error: {rmse_mult:.2f}")
print(f"Mean Absolute Error: {mae_mult:.2f}")

print(f"\nFeature Coefficients (after scaling):")
for feature, coef in zip(feature_columns, multiple_model.coef_):
    print(f"{feature}: {coef:.4f}")
print(f"Intercept: {multiple_model.intercept_:.2f}")


In [ ]:
# Visualize multiple regression results
plt.figure(figsize=(12, 5))

# Plot 1: Actual vs Predicted
plt.subplot(1, 2, 1)
plt.scatter(y_test_mult, y_pred_mult, color='blue', alpha=0.5)
plt.plot([y_test_mult.min(), y_test_mult.max()], [y_test_mult.min(), y_test_mult.max()], color='red', linestyle='--', linewidth=2)
plt.xlabel('Actual Total Passengers')
plt.ylabel('Predicted Total Passengers')
plt.title('Actual vs Predicted - Multiple Regression')
plt.grid(True, alpha=0.3)

# Plot 2: Residual plot
plt.subplot(1, 2, 2)
residuals_mult = y_test_mult - y_pred_mult
plt.scatter(y_pred_mult, residuals_mult, color='green', alpha=0.5)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot - Multiple Regression')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Model Comparison and Analysis

In [ ]:
# Compare the performance of both models
print("Model Comparison:")
print("=" * 70)
print(f"{'Metric':<25} {'Simple Regression':<20} {'Multiple Regression':<20}")
print("=" * 70)

print(f"{'R² Score':<25} {r2_simple:<20.4f} {r2_mult:<20.4f}")
print(f"{'RMSE':<25} {rmse_simple:<20.2f} {rmse_mult:<20.2f}")
print(f"{'MAE':<25} {mae_simple:<20.2f} {mae_mult:<20.2f}")

print("=" * 70)

if r2_mult > r2_simple:
    better_model = "Multiple Regression"
    improvement = ((r2_mult - r2_simple) / r2_simple) * 100
else:
    better_model = "Simple Regression"
    improvement = ((r2_simple - r2_mult) / r2_mult) * 100

print(f"\nBest Model: {better_model}")
print(f"R² Improvement: {improvement:.2f}%")


## 7. Statistical Insights and Conclusions

In [ ]:
# Summarize your findings and provide insights
print("STATISTICAL INSIGHTS AND CONCLUSIONS")
print("=" * 60)

print("\n1. HYPOTHESIS TESTING RESULTS:")
print(f"   • Domestic vs International Passengers: There is a statistically significant difference between the two groups. Domestic passengers outnumber international ones.")
print(f"   • Correlation between Total Passengers and Flights: There is a highly significant positive correlation, meaning more flights directly translate to more passengers.")

print("\n2. REGRESSION ANALYSIS:")
print(f"   • Simple Linear Regression R²: {r2_simple:.4f} (Indicates the percentage of variance in passengers explained purely by total flights)")
print(f"   • Multiple Linear Regression R²: {r2_mult:.4f} (Indicates a perfect fit, likely due to data leakage since features inherently comprise the target variable 'Pax')")
print(f"   • Best performing model: Multiple Regression (but note the risk of overfitting/data leakage).")

print("\n3. KEY FINDINGS:")
print(f"   • Domestic traffic forms the bulk of the air traffic, both in terms of flights and passengers.")
print(f"   • Total flights serve as a very strong predictor of total passengers, though the capacity per flight adds variance.")
print(f"   • The multiple regression model achieved a perfect or near-perfect R² because 'Pax' is the sum of 'Dom_Pax' and 'Int_Pax'.")

print("\n4. RECOMMENDATIONS:")
print("   • Focus resource allocation on domestic terminals since they handle the significantly larger share of passenger volume.")
print("   • Use the simple linear regression model for high-level forecasting when only flight schedule data is available.")
print("   • When building future predictive models, avoid including deterministic components to prevent data leakage and evaluate true underlying trends.")


## 8. Reflection Questions

**Answer the following questions based on your analysis:**

1. **Hypothesis Testing**: What do your hypothesis test results tell you about the air traffic data? Were the results expected?

   *Answer: Our hypothesis test confirms there is a statistically significant difference between the average number of domestic passengers and international passengers. Additionally, the correlation test confirms a very strong relationship between the number of flights and the number of passengers, which is entirely expected since flights carry passengers.*

2. **Model Performance**: Which regression model performed better and why? What does the R² value tell you?

   *Answer: The multiple regression model vastly outperformed the simple linear regression model (with an R² close to 1.0). This happened primarily because the multiple regression features included "Dom_Pax" and "Int_Pax," whose sum exactly equals the target "Pax". The R² tells us what percentage of the variance in the target variable is captured by the model.*

3. **Correlations**: What were the strongest correlations you found? How might these relationships be useful for airlines?

   *Answer: The strongest correlations were between Pax and Dom_Pax, and Pax and Flt. This makes sense as domestic traffic makes up the majority of flights. Airlines can use these correlations to proxy total revenue purely from domestic trends without needing to wait for complete international datasets.*

4. **Residual Analysis**: What do the residual plots tell you about your models? Are there any patterns that suggest model improvements?

   *Answer: The simple linear regression model's residuals show a scattered pattern, reflecting natural variance because larger flight volumes inherently carry larger variations in passenger loads. The multiple regression model's residuals are essentially zero, highlighting data leakage.*

5. **Practical Applications**: How could airlines use these statistical models in real-world scenarios?

   *Answer: Airlines can utilize simple linear regression on scheduled flight frequencies to estimate terminal crowdedness and staff ground operations accordingly. However, they should avoid using direct subcomponents to predict a total, instead leveraging external data (like ticket prices or GDP) for more realistic predictive forecasting.*
